# Lesson 9 · Every frame of sound is an arrow — vector addition, scaling, and linear combination

**Learning goals**
1. Understand that a vector is a NumPy array of shape `(n,)`, and that `.shape` is the attribute you always check
2. Master the algebraic and geometric meaning of vector addition (mixing) and scalar scaling (turning the volume up or down)
3. Implement a `scale(v, c)` function and verify scalar multiplication with a unit test
4. Compute a vector's length (Pythagorean theorem / `np.linalg.norm`), and know that one MFCC frame is a 13-dimensional vector
5. Understand linear combination: any 2D vector can be written as a weighted sum of the basis vectors `e1`, `e2`

**Why it matters for Aurora**: Aurora's MFCC feature extraction squeezes each audio frame into a `(13,)` vector; when `scale(v, c)` adjusts the volume, it is doing scalar scaling on exactly that vector. The activations of every layer in a neural network are vectors too.

← **Previous lesson** · [L08 · Visualizing the complex plane](../1_complex_trig/L08_visual_complex.ipynb)

> Last lesson covered **visualizing the complex plane**: rotation on the unit circle, conjugates and phase, with animated matplotlib demos.
> This lesson explores **vector algebra**.

## This lesson's story: a list of numbers hiding a direction and a size

You're at the mixing desk and you push up a fader — the sound gets louder. In math terms: every sample value is multiplied by the same factor. That is **scalar multiplication**.

Now layer two audio tracks — a male voice and a female voice playing at the same time. In math terms: two arrays of the same length are added element by element. That is **vector addition**.

Those two things are all the basic operations a vector has — **this lesson practices just these two bricks**: addition and scaling (and then builds linear combination out of them).

The foundation of the linear-algebra building is the vector. Matrices, eigen-directions and SVD show up in later lessons; today you don't have to put up the whole building at once.

In this lesson you'll first write `scale` and `add_signals`, then use them to build `linear_combination`.

## 1. A vector is just a numpy array

Ever wondered why linear algebra ends up using NumPy?

Because a one-dimensional NumPy array *is* a vector — an **ordered list of numbers** `[x₁, x₂, …, xₙ]` with shape `(n,)`.

Physically, a vector is "an arrow with a direction":
- The 2D vector `[3, 4]` → an arrow from the origin, 3 to the right and 4 up, with length 5
- An audio signal `[0.1, -0.3, 0.5, ...]` → a point (or a path) in N-dimensional space

Vectors are everywhere in audio processing:
- One frame of MFCC coefficients = a 13-dimensional vector (a point in feature space)
- The high-dimensional features you'll meet later (say, a "semantic vector" written as a very long list of numbers) = exactly the same rules, just more components
- The similarity between two audio frames = a vector dot product

**Key intuition**: `shape = (n,)` is a vector, `shape = (m, n)` is a matrix (several vectors stacked). Read the shape carefully and you won't get lost in matrix multiplication.

### Before going abstract: how does an arrow turn into an array?

The vectors you met in high school were "arrows" from the origin to some point on the plane — "walk 3 blocks east from home and 4 blocks north," and there's your arrow. But in Aurora a vector is a list of numbers `[0.1, -0.3, 0.5, ...]` that looks nothing like an arrow. Are these really the same thing?

**Key insight**: the only reason we can draw the arrow at all is that we described it with a set of numbers like "3 to the right, 4 up" — the pair `(3, 4)` *is* all the information in the arrow. Drawing the arrow is just one way of "translating" those numbers into a picture on paper. **The numbers are the vector itself**; the arrow is merely how it looks when we visualize it in 2D or 3D.

**What about an audio signal?** `[0.1, -0.3, 0.5, 0.2, -0.4]` is 5 numbers, and algebraically it corresponds to an "arrow" in 5-dimensional space — from the origin to the point with coordinates `(0.1, -0.3, 0.5, 0.2, -0.4)`. We can't draw a 5-dimensional arrow on paper, but it follows exactly the same rules as a 2D arrow: addition is position-by-position addition, scaling is position-by-position multiplication.

**"Point" or "arrow" — which one should I say?** The same array can be seen as "the point at these coordinates" or as "the arrow from the origin to that point" — both describe the same mathematical object. "Arrow" emphasizes direction/displacement; "point" emphasizes position. In engineering you needn't agonize over the word: calling an MFCC vector "a point in feature space" or "an arrow" are both correct — pick whichever matches whether you want to stress "where it is" or "where it points."

**Once the dimension climbs, don't count on pictures**: past 3 dimensions the human brain can't draw arrows anymore, and that's perfectly fine — the two basic vector operations (addition and scaling) never depended on being drawable; they are defined as **component-wise operations**. For a 13-dimensional MFCC vector you don't need to force yourself to imagine 13-dimensional space; just remember: "vector arithmetic = doing the same arithmetic independently on each number in the list." That's also why NumPy code for a 3-dimensional vector looks exactly like the code for a 300-dimensional one — we'll check this directly with a 5-dimensional example below.

## Notation first: look at the shape, then the operation

Every object in linear algebra has a definite shape: a vector is `(n,)`, a matrix is `(m, n)`, and a matrix times a vector turns `(n,)` into `(m,)`. Each example starts by marking the input, the transformation, and the output.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
v = np.array([3.0, 4.0])      # a 2-dimensional vector
audio = np.array([0.0, 0.5, 1.0, 0.5, 0.0])  # 5 samples = a 5-dimensional vector
print('v =', v, '| dimension:', v.shape)
print('audio is a', audio.shape[0], '-dimensional vector')

## Hands-on: shape and norm

Look at `v.shape` printing `(2,)`, `A.shape` printing `(2, 2)`, and the result of `A @ v` — the matrix stretches the vector: the shape stays the same but the values change. `np.linalg.norm(v)` returns 5.0, i.e. √(3²+4²); this is how NumPy writes a vector's length.

In [ ]:
import numpy as np

v = np.array([3.0, 4.0])
A = np.array([[2.0, 0.0],
              [0.0, 0.5]])

print('v =', v, 'shape =', v.shape)
print('A =')
print(A)
print('A shape =', A.shape)
print('A @ v =', A @ v)
print('vector length ||v|| =', np.linalg.norm(v))


### A footnote: where does the `‖v‖` symbol come from? And the formula?

Start with an everyday scene: you're on a sports field, you walk 3 meters east, then 4 meters north. How far are you from your starting point in a straight line? Sketch it and you'll see a right triangle with legs 3 and 4, and you want the hypotenuse. The Pythagorean theorem from middle school tells you: hypotenuse² = 3² + 4² = 25, so the hypotenuse = 5.

**That is exactly where the vector-length formula comes from**: the "length" of the vector `[3, 4]` is the straight-line distance from the origin to the point `(3, 4)`, and that distance is precisely the hypotenuse in the Pythagorean theorem. So the formula `‖v‖ = √(v₁² + v₂²)` isn't an arbitrary rule — it's the Pythagorean theorem in different clothing, with "the two legs" replaced by "the two components of the vector." Negative components (like `[3, -4]`) don't matter either, because every component is squared first and the minus sign disappears: geometrically, "walked 4 meters" and "which direction you walked 4 meters" don't change the distance itself.

**Why a double bar `‖v‖` rather than the single bar `|x|` of absolute value?** Because a vector has "many components" where a scalar has one, and the double bar reminds you that you're measuring the "overall size" of the whole vector, not the absolute value of a single number. A tool for measuring this "overall size" is called a **norm** in mathematics — the word comes from the Latin *norma*, originally "a standard, a measuring rule": a norm is a ruler for measuring how big a vector is.

**Is there only one way to use this ruler?** No. `√(v₁² + v₂²)` (square, sum, take the square root) is the most common one, called the **L2 norm**, and it corresponds to the familiar "straight-line distance" (Euclidean distance). There's an even plainer way: just add up the absolute values of the components, `|v₁| + |v₂|`. That's the **L1 norm**, corresponding to "taxicab distance" (Manhattan distance) — picture a city laid out on a grid where a taxi can't cut diagonally and has to follow the streets; the total distance it drives is the L1 norm. Both "rulers" are legitimate; they just care about different things: L2 cares about the straight-line distance between two points, L1 about the total distance traveled "along the axes," and L1 is also less sensitive to a single component suddenly becoming very large (an outlier). `np.linalg.norm(v)` computes L2 by default; pass `ord=1` to switch to L1 — let's run both and compare.


In [ ]:
v = np.array([3.0, 4.0])
l2 = np.linalg.norm(v)          # default ord=2: Euclidean distance (straight-line distance)
l1 = np.linalg.norm(v, ord=1)   # ord=1: sum of absolute values (taxicab distance)
print(f'L2 norm ‖v‖₂ = √(3²+4²)   = {l2}')
print(f'L1 norm ‖v‖₁ = |3| + |4|  = {l1}')


## Code experiment: every vector is a linear combination of basis vectors

The code below shows that **every 2D vector is a linear combination of the basis vectors e₁, e₂**: given coefficients (a, b), the output vector is just `a·e₁ + b·e₂`. This is the most direct geometric meaning of a linear combination — the coefficients decide the direction and length, and the basis vectors provide the "coordinate axes."

> **Note**: what a matrix does to a vector is covered in detail in L12 (matrices).

In [ ]:
import numpy as np

# Linear combination: 2 basis vectors added together with different coefficients
e1 = np.array([1.0, 0.0]); e2 = np.array([0.0, 1.0])
for a, b in [(1,0), (0,1), (2,-1), (0.5, 1.5)]:
    v = a*e1 + b*e2
    print(f'{a}*e1 + {b}*e2 = {v}')
print('→ any 2D vector = coeff1*e1 + coeff2*e2')


### One more question: why "any vector"? Would a different set of basis vectors work?

The code above only shows that "a few examples worked," which isn't a proof. Think of it another way: when you give a friend directions, "go so many blocks east and so many blocks north" gets them to any intersection in town — provided that "due east" and "due north" are **independent of each other**: no matter how far you go east, it never nudges you north along the way. This "neither can be built from the other" relationship is called **linear independence** in mathematics. Any set of directions whose linear combinations can reach every point in the plane is called a **basis**. `e1=[1,0]`, `e2=[0,1]` is just the most convenient basis, not the only one.

**Why does a 2D plane need exactly 2 basis vectors, no more and no less?** Because "2-dimensional" itself means "you need exactly 2 mutually independent directions to reach every point in the plane." Give only 1 direction and you can only move along a line, never reaching points off it; give 3 directions and one of them can necessarily be written as a linear combination of the other two (not independent — redundant). Number of dimensions = the minimum number of independent directions needed. That's no coincidence; it's how the word "dimension" is defined.

**Can other vectors serve as a basis?** As long as the two vectors aren't parallel (don't lie on the same line), they're still independent and can still be a basis — below we try to express the same target vector with `f1=[1,1]`, `f2=[1,-1]` and see whether it works. Conversely, if two vectors are parallel (say `[1,1]` and `[2,2]`, the second is just twice the first), they can only cover a single line and can't reach most points in the plane — then the "basis" fails, and the technical term is that the two vectors are **linearly dependent**. The code below runs both cases.


In [ ]:
target = np.array([3.0, 4.0])

# A different basis: f1, f2 are not parallel -> still a valid basis
f1 = np.array([1.0, 1.0]); f2 = np.array([1.0, -1.0])
basis_matrix = np.column_stack([f1, f2])          # the two columns are f1, f2
a, b = np.linalg.solve(basis_matrix, target)      # solve for coefficients a, b with a*f1 + b*f2 = target
print(f'{a}*f1 + {b}*f2 = {a*f1 + b*f2}  (target is {target})')

# Counterexample: g1, g2 are parallel (g2 = 2*g1), linearly dependent, the basis fails
g1 = np.array([1.0, 1.0]); g2 = np.array([2.0, 2.0])
try:
    bad_basis = np.column_stack([g1, g2])
    np.linalg.solve(bad_basis, target)
except np.linalg.LinAlgError as e:
    print(f'g1, g2 are parallel and cannot form a basis: {e}')


## 2. Addition = element-wise sum; scaling = multiply every element by the same number

In [ ]:
a = np.array([1.0, 2.0])
b = np.array([3.0, 1.0])
print('a + b =', a + b)     # layering signals (mixing) is vector addition
print('2 * a =', 2 * a)     # amplifying 2x = scalar scaling

## 3. ✏️ Your task: implement `scale`

`scale(v, c)` multiplies every component of vector `v` by the scalar `c` — this is the mathematical essence of a volume knob.

**Line of reasoning**:
1. The geometric meaning of scalar scaling: the vector's direction stays the same and its length becomes `|c|` times the original; when `c < 0` the direction flips, and when `c = 0` it becomes the zero vector.
2. NumPy broadcasting: `c * v` automatically applies the scalar `c` to every element of `v` — no for loop needed; under the hood it uses SIMD parallel instructions.
3. Returned shape: `c * v` has exactly the same shape as `v`, no extra reshape needed.

**Reference input and output**: `scale([1.0, 2.0], 2.0)` → `[2.0, 4.0]`; `scale([0.2, -0.4], 1.5)` → `[0.3, -0.6]`

> 💡 Need a hint? Once you've finished the exercise, you can check the reference implementation in the `solutions/` folder.


### When `c < 0` "flips the direction," what does that mean for a 13-dimensional MFCC vector?

Multiply the 2D vector `[3, 4]` by `-1` and you get `[-3, -4]`; draw it on paper and it's obvious: the arrow has turned 180 degrees and points the opposite way. But in Aurora, `scale(frame, -1)` acts on a 13-dimensional MFCC vector, and all 13 numbers change sign — no sheet of paper can show that "arrow," so does "opposite direction" still mean anything?

**It does, but now the meaning is purely algebraic and no longer depends on being drawable.** The heart of "pointing the opposite way" isn't really the picture of an arrow turning 180 degrees; it's two properties you can check directly with numbers:
1. Same size — `‖v‖ = ‖-v‖`, the length is the same before and after the flip;
2. They cancel out — `v + (-v)` is exactly the all-zero vector; the two "cancel each other completely."

Both properties involve only component-wise addition and negation, so they hold equally in 2D, 3D, 13 dimensions or 300 dimensions — they don't need to be "drawable" to count. So you can think of "direction" in high-dimensional space like this: the abstract algebraic structure is preserved intact; only the accompanying "arrow picture" is gone. Next lesson's dot product will give you an even stronger tool — even when you can't picture anything at all, you can compute the "angle" between any two high-dimensional vectors, and then the word "direction" becomes more precise and more computable.

For now, remember just one sentence: a 13-dimensional `scale(frame, -1)` flips the sign of each of the 13 numbers — in audio engineering this is equivalent to "phase inversion," commonly used for cancellation in noise reduction. Below we verify directly that the two properties above still hold for a 13-dimensional vector.


In [ ]:
frame = np.array([0.2, -0.1, 0.05, 0.3, -0.2, 0.1, 0.0, 0.15, -0.05, 0.2, -0.1, 0.05, 0.1])  # a simulated 13-dimensional MFCC frame
flipped = -1 * frame  # equivalent to scale(frame, -1): each of the 13 components changes sign

print('original length ‖frame‖   =', np.linalg.norm(frame))
print('flipped length  ‖flipped‖ =', np.linalg.norm(flipped), '  (same size)')
print('frame + flipped     =', frame + flipped, '  (they cancel out: all-zero vector)')


### Three things to pin down before writing `scale`

- Input: `v` (numpy array, any shape); `c` (float, the scaling factor)
- Key step: `c * v` — NumPy's scalar multiplication acts on every element equally
- Returns: an array with the same shape as `v`, each element `c` times its original value

In [ ]:
def scale(v, c):
    # ✏️ TODO: return v scaled by c
    raise NotImplementedError("TODO: implement scale — hint: use c * v")


### While we're at it: why does the check below use `np.allclose` instead of `==`?

The verification code you're about to run contains `np.allclose(louder, [0.3, -0.6, 0.9, -1.2])` rather than simply `louder == [0.3, -0.6, 0.9, -1.2]`. That looks like taking the long way round — shouldn't `[0.2,-0.4,0.6,-0.8]` times `1.5` come out to exactly `[0.3,-0.6,0.9,-1.2]`?

Here's an analogy: you take a ruler that can only be read precisely to the millimeter and measure a wooden strip that is, in theory, exactly 30 centimeters long. The reading might be 299.9999 millimeters, or 300.0001 millimeters — the ruler isn't broken; its precision only goes down to millimeters and it can't measure anything finer. Computers store decimals as **floating point** numbers, which are essentially a "number ruler" of limited precision too: the vast majority of decimal fractions (including such "clean"-looking numbers as `0.1` and `0.3`) simply cannot be stored exactly in the computer's internal binary representation — only an extremely close approximation can be stored.

The classic example: compute `0.1 + 0.2` directly in Python and you'll find the result isn't `0.3` but `0.30000000000000004` — that tiny extra tail is the fingerprint of floating-point precision. So when you check "are these two floating-point results equal," you must never demand an exact match (`==`); use `np.allclose(a, b, atol=1e-10)` instead — meaning "as long as the gap between them is smaller than the tiny number `atol` (absolute tolerance), treat them as equal." This isn't laziness or cutting corners; it's the universally accepted standard practice in numerical code. Let's run it and see.


In [ ]:
print('0.1 + 0.2          =', 0.1 + 0.2)
print('0.1 + 0.2 == 0.3 ?  ', 0.1 + 0.2 == 0.3)                    # False: floating-point error
print('np.allclose(0.1+0.2, 0.3) ?', np.allclose(0.1 + 0.2, 0.3))  # True: equal within tolerance


In [ ]:
try:
    audio = np.array([0.2, -0.4, 0.6, -0.8])
    louder = scale(audio, 1.5)
    print('original:', audio)
    print('louder:', louder)
    assert np.allclose(louder, [0.3, -0.6, 0.9, -1.2]), 'every element should be ×1.5'
    print('\n✅ Passed: you can scale a vector = adjust the volume.')
except (NotImplementedError, TypeError) as e:
    print(f'⚠️  Not implemented yet: {e}')
    print('Please fill in the body of scale(), then re-run this cell.')


## ✏️ Exercise A: implement `add_signals` (vector addition = mixing)

What vector addition means for audio: layering two sounds sample by sample.
**Line of reasoning**: `a[i] + b[i]` for every i, which is equivalent to NumPy's `a + b`.
- Input: two numpy arrays `a`, `b` of the same length
- Output: their element-wise sum, with the same shape as the inputs

In [ ]:
def add_signals(a, b):
    # ✏️ TODO: return the element-wise sum of a and b
    raise NotImplementedError("TODO: implement add_signals — hint: use a + b")


try:
    ch1 = np.array([0.1, 0.3, -0.2])
    ch2 = np.array([0.4, -0.1, 0.5])
    mix = add_signals(ch1, ch2)
    assert np.allclose(mix, [0.5, 0.2, 0.3]), 'element-wise sum is wrong'
    assert mix.shape == ch1.shape, 'shape must match the input'
    print('✅ add_signals passed')
except (NotImplementedError, TypeError) as e:
    print(f'⚠️  Not implemented yet: {e}')


## ✏️ Exercise B: implement `linear_combination`

A linear combination folds a set of `(coefficient, vector)` pairs into a single vector:
`coeffs[0]*vecs[0] + coeffs[1]*vecs[1] + … + coeffs[k-1]*vecs[k-1]`

- Input: `coeffs` (a list of k floats), `vecs` (a list of k numpy arrays of equal length)
- Output: one numpy array with the same shape as `vecs[0]`

In [ ]:
def linear_combination(coeffs, vecs):
    # ✏️ TODO: add up all the coeffs[i] * vecs[i]
    raise NotImplementedError("TODO: implement linear_combination — hint: use a running sum")


try:
    e1 = np.array([1.0, 0.0]); e2 = np.array([0.0, 1.0])
    # 2·e₁ + (-1)·e₂ should equal [2, -1]
    result = linear_combination([2, -1], [e1, e2])
    assert np.allclose(result, [2.0, -1.0]), 'linear combination result is wrong'
    # zero-coefficient edge case: 0·e₁ + 0·e₂ = zero vector
    zero = linear_combination([0, 0], [e1, e2])
    assert np.allclose(zero, [0.0, 0.0]), 'all-zero coefficients should return the zero vector'
    print('✅ linear_combination passed')
except (NotImplementedError, TypeError) as e:
    print(f'⚠️  Not implemented yet: {e}')


### What is linear combination actually good for? — More than a math exercise

Working out `2·e1 + (-1)·e2 = [2, -1]` algebraically isn't hard, but what does it mean in practice? Why go out of our way to "break" a vector into a weighted sum of basis vectors?

Picture the equalizer (EQ) on a mixing desk: it has a row of faders, each controlling the volume of one frequency band. The sound that finally comes out of the speakers is really "each band's standard waveform, multiplied by its fader position, all added together" — and that's exactly a linear combination: `coeffs[i] * vecs[i]`, multiplied term by term and summed. Each band's standard waveform plays the role of a "basis vector," and the fader positions are the "coefficients." The reverse holds too: any complex sound (or image, or any feature vector) can be seen as "a few simple, standard directions" stacked up with different weights — this idea of "breaking the complex into the simple and stacking with weights" will come back again and again in Fourier decomposition and the MFCC feature basis later on. It's a core idea that signal processing and machine learning share.

A concrete example in Aurora's setting: suppose the male voice and the female voice have already been recorded as two separate tracks (two vectors), and you want to turn the male voice down and the female voice up. The mixed output is:


In [ ]:
try:
    male_track   = np.array([0.2, -0.1, 0.15])
    female_track = np.array([0.05, 0.3, -0.2])
    # Male voice at 0.5x volume, female voice at 1.5x volume, then mix — that's one linear combination
    mixed = linear_combination([0.5, 1.5], [male_track, female_track])
    print('mixed result =', mixed)
except (NotImplementedError, TypeError) as e:
    print(f'⚠️  Please finish linear_combination() above first: {e}')


## 4. Geometry: a vector is an arrow with a direction

In [ ]:
import matplotlib.pyplot as plt
v = np.array([3.0, 4.0])
plt.figure(figsize=(4,4))
plt.quiver(0,0, v[0], v[1], angles='xy', scale_units='xy', scale=1)
plt.xlim(-1,5); plt.ylim(-1,6); plt.grid(True, alpha=.3)
plt.gca().set_aspect('equal'); plt.title('vector [3,4]'); plt.show()

### One more comparison plot: negative scaling = a 180-degree turn

The line of reasoning above said "when `c < 0` the direction flips," but the plot above only drew the single arrow `[3, 4]`, not what it looks like after flipping. It's hard to picture "turned 180 degrees" from words alone, so below we draw `v=[3,4]` and `-v=[-3,-4]` on the same plot: you'll see two arrows pointing in exactly opposite directions with exactly the same length, like a clock hand swung around to the opposite side.


In [ ]:
v = np.array([3.0, 4.0])
neg_v = -1 * v
plt.figure(figsize=(4,4))
plt.quiver(0,0, v[0], v[1], angles='xy', scale_units='xy', scale=1, color='tab:blue')
plt.quiver(0,0, neg_v[0], neg_v[1], angles='xy', scale_units='xy', scale=1, color='tab:red')
plt.xlim(-5,5); plt.ylim(-5,6); plt.grid(True, alpha=.3)
plt.gca().set_aspect('equal')
plt.title('v=[3,4] (blue) and -v=[-3,-4] (red): opposite direction, same length')
plt.show()


**🔗 Aurora connection**: the `mfcc()` function in `aurora/audio/mfcc.py`
returns a matrix of shape `(n_frames, 13)`, where each row is the 13-dimensional MFCC coefficient vector of one audio frame;
take one row (e.g. `coeffs[i]`, shape `(13,)`) and scale it element-wise with `scale(frame, gain)`,
and that is the mathematical essence of Aurora's volume-normalization step (adjusting volume = multiplying by a scalar, without changing the spectral shape).
Next lesson's dot product (L10) will let you quantitatively compare the similarity of two MFCC frame vectors — the underlying operation of speech retrieval and attention mechanisms.

### Pitfall alert: `mfcc[i]` and `mfcc[i:i+1]` are not the same thing

Above we said "take one row `coeffs[i]`, with shape `(13,)`" — and here's a trap beginners fall into easily: if you accidentally write `mfcc[i]` as `mfcc[i:i+1]`, the two hold what looks like the same data but have different shapes, and later matrix operations may throw mysterious errors or give wrong results.

An analogy: a class roll book is like an `(n_frames, 13)` matrix, where each row is a student (a vector). "Calling the 3rd student's name" (corresponding to `mfcc[2]`) and "cutting the 3rd row out of the roll book as a separate mini roll book with only one row" (corresponding to `mfcc[2:3]`) both give you the same student's information, but the first hands you "this one student in person" (one dimension dropped: directly a `(13,)` vector), while the second still hands you "a roll book" that just happens to have only one row (two dimensions kept: shape `(1, 13)`).

**Remember this rule**: a single integer inside the brackets (like `mfcc[2]`) → that dimension gets "eaten" and the number of dimensions drops by one; a slice inside the brackets (like `mfcc[2:3]`, or the `:` in `mfcc[2, :]`) → the dimension is kept. `mfcc[i, :]` and `mfcc[i]` are actually equivalent and both drop a dimension; only the range slice `i:i+1` keeps that "one-row" two-dimensional shape. Below we verify the shape differences of the three forms directly on a small simplified matrix.


In [ ]:
mfcc_like = np.array([[0.1, 0.2, 0.3],
                       [0.4, 0.5, 0.6],
                       [0.7, 0.8, 0.9]])   # pretend this is a simplified (3 frames, 3 dims) MFCC matrix

row_index = mfcc_like[1]        # integer index: drops a dimension -> (3,)
row_slice = mfcc_like[1:2]      # range slice: keeps two dimensions -> (1,3)
row_colon = mfcc_like[1, :]     # equivalent to mfcc_like[1], also drops a dimension -> (3,)

print('mfcc_like[1]    shape =', row_index.shape, '->', row_index)
print('mfcc_like[1:2]  shape =', row_slice.shape, '->', row_slice)
print('mfcc_like[1, :] shape =', row_colon.shape, '->', row_colon)
print('Same values?', np.allclose(row_index, row_slice.flatten()))


## 🎨 Diagram: vector = an arrow with a direction, scaling = changing its length

In [ ]:
from laviz import style, arrows2d
style()
arrows2d([[3,4],[1,2],[6,8]], ['a=[3,4]','b=[1,2]','2a (scaled)'],
         title='Vectors and scaling');

### First, a quick intuition: what is `A @ v` actually doing?

The full theory of how a matrix acts on a vector is saved for L12, but since the code coming up uses `A @ v` right away, here's an intuition to tide you over until then, so it isn't left "hanging in the air."

Remember what we just learned: any vector can be written as a linear combination of basis vectors, e.g. `v = v₁·e1 + v₂·e2`. Each column of the matrix `A` is really telling you "where the original e1 and e2 have been moved to." For example, with `A = [[2,1],[0,1]]`, its first column `[2,0]` is "where the new e1 lands," and its second column `[1,1]` is "where the new e2 lands."

So what `A @ v` does is: first break `v` apart, as a linear combination, into `v₁` parts e1 plus `v₂` parts e2; then, instead of the original e1 and e2, use "the new e1 and new e2 that A gives you," and reassemble with the same coefficients `v₁`, `v₂` — that is, `A @ v = v₁ · (A's first column) + v₂ · (A's second column)`. The reason a matrix can "stretch, rotate, and shear" a vector is, at heart, that it quietly moves the basis vectors around; and since every vector is just a linear combination of the basis vectors, once the basis vectors move house, the combined result moves with them. Below we verify that this equation holds; the full story of "how a matrix determines stretching directions, rotation angles, and eigenvectors" comes in L12.


In [ ]:
A_demo = np.array([[2.0, 1.0], [0.0, 1.0]])
v_demo = np.array([1.0, 1.0])

lhs = A_demo @ v_demo
rhs = v_demo[0]*A_demo[:, 0] + v_demo[1]*A_demo[:, 1]   # linear combination of A's two columns
print('A @ v            =', lhs)
print('v0*col0 + v1*col1 =', rhs)
print('Equal:', np.allclose(lhs, rhs))


In [ ]:
A = np.array([[2.0, 1.0], [0.0, 1.0]])
probes = np.array([[1,0], [0,1], [1,1], [-1,2]], dtype=float)
print('How does matrix A move these vectors?')
for v in probes:
    out = A @ v
    print(f'{v} -> {out} | length {np.linalg.norm(v):.2f} -> {np.linalg.norm(out):.2f}')


## Parameter experiment: turn one knob at a time

1. Change `A[0][1]` from `1.0` to `0.0` — the matrix becomes diagonal, and the x component is no longer affected by y; check: only the off-diagonal entries make different components influence each other.
2. Change `A[0][1]` to a negative number (e.g. `-2.0`), work out the output for `v=[1, 1]` by hand first, then run it and compare — a negative value "pulls the x component the opposite way."
3. Replace one vector in `probes` with `[3.0, 4.0]` (norm = 5.0) and see whether the matrix scales this "long" vector by the same factor as `[1, 1]` (hint: a matrix transformation generally does **not** preserve the norm, unless it's an orthogonal matrix).

## Wrap-up

You can now use `scale(v, c)` to scale any numpy array element-wise, with an output shape exactly the same as the input. This corresponds to the volume adjustment in Aurora's `aurora/audio`, while vector addition corresponds to mixing — together the two operations cover the underlying logic of signal synthesis. MFCC feature extraction ultimately squeezes each frame into a `(13,)` vector, which is exactly the kind of object you worked with today. Next lesson's dot product will let you compute the similarity of two vectors, the basic operation behind feature matching and attention mechanisms.

## ✏️ Whiteboard challenge: vector arithmetic by hand (target: 8 minutes)

Cover the screen and answer on paper:

**Question 1**: v = [3, 4], c = 2.5, `scale(v, c)` = ?

**Question 2**: a = [1.0, -0.5, 0.8], b = [0.2, 0.7, -0.3], `add_signals(a, b)` = ?

**Question 3**: coeffs = [2.0, -1.0, 0.5], vecs = [[1,0],[0,1],[2,2]]
`linear_combination(coeffs, vecs)` = 2×[1,0] + (−1)×[0,1] + 0.5×[2,2] = ?

**Question 4**: What is the L2 norm (Euclidean length) of the vector [3, 4]? Formula: `‖v‖ = √(v₁² + v₂²)` = ?

When you've finished, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

passed = []

# Q1: scale
v, c = np.array([3.0, 4.0]), 2.5
q1_expected = np.array([7.5, 10.0])
try:
    q1 = scale(v, c)
    assert np.allclose(q1, q1_expected, atol=1e-10), f"scale result {q1} vs expected {q1_expected}: mismatch"
    print(f"Q1 ✅  scale([3,4], 2.5) = {q1}")
    passed.append(1)
except (NotImplementedError, TypeError):
    print("⬜ Q1: implement scale() first, then run the answer-check cell")

# Q2: add_signals
a, b = np.array([1.0, -0.5, 0.8]), np.array([0.2, 0.7, -0.3])
q2_expected = np.array([1.2, 0.2, 0.5])
try:
    q2 = add_signals(a, b)
    assert np.allclose(q2, q2_expected, atol=1e-10)
    print(f"Q2 ✅  add_signals = {q2}")
    passed.append(2)
except (NotImplementedError, TypeError):
    print("⬜ Q2: implement add_signals() first, then run the answer-check cell")

# Q3: linear_combination
coeffs = [2.0, -1.0, 0.5]
vecs   = [np.array([1.0, 0.0]), np.array([0.0, 1.0]), np.array([2.0, 2.0])]
q3_expected = np.array([3.0, 0.0])
try:
    q3 = linear_combination(coeffs, vecs)
    assert np.allclose(q3, q3_expected, atol=1e-10)
    print(f"Q3 ✅  linear_combination = {q3}")
    passed.append(3)
except (NotImplementedError, TypeError):
    print("⬜ Q3: implement linear_combination() first, then run the answer-check cell")

if len(passed) == 3:
    print("\n🎉 Whiteboard challenge passed! The three vector operations are now second nature.")


In [ ]:
# ✏️ Self-review for this lesson
l09_review = {
    "scale_implemented":             None,  # scale implemented and passes its assert? True/False
    "add_signals_implemented":       None,  # add_signals implemented and passes its assert? True/False
    "linear_combination_implemented": None, # linear_combination implemented and passes its assert? True/False
    "vector_as_arrow_intuition":     None,  # understand "vector = an arrow with a direction"? True/False
    "whiteboard_passed":             None,  # finished the whiteboard challenge on paper? True/False
}

unfilled = [k for k, v in l09_review.items() if v is None]
assert not unfilled, f'Not filled in yet: {unfilled}'
weak = [k for k, v in l09_review.items() if v is False]
if weak:
    print(f'⚠️  Needs more work: {weak}')
else:
    print('✅ L09 complete! On to L10: dot product and projection')

---

→ **Next lesson** · [L10 · Dot product and projection](L10_dot_product.ipynb)

> Next lesson covers **dot product and projection**: a·b = |a||b|cosθ, and why similarity = dot product ÷ product of the norms.